# Demostración del funcionamiento de la librería para la detección anticipada de riesgos con razonadores sobre flujos

Clonamos el repositorio principal e instalamos las herramientas principales. Luego de ejecutar la siguiente celda de código hay que reiniciar la sesión de Google Colab (``Runtime -> restart session``) en un entorno con **GPU**, y retomar la ejecución del cuaderno desde la segunda celda de código.

In [2]:
!git clone https://github.com/matizzat/social-media-streams.git
%cd social-media-streams
%pip install -e .
%pip install llms_kgs
%pip install ollama FlagEmbedding pgVector pyvis lorem_text

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.8 MB/s eta 0:00:00


Instalamos Ollama, comenzamos su ejecución y descargamos el modelo Gemma3 12B.

In [1]:
!sudo apt-get install zstd
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

!ollama pull gemma3:12b

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (568 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently i

Importamos las librerías necesarias

In [2]:
from social_media.classifier.discourse_element_classifier import DiscourseElementClassifier
from social_media.classifier.prompts import DISCOURSE_ELEMENT_CLASSIFIER_SYSTEM_TEMPLATE

from social_media.erisk.discourse_element_provider import DiscourseElementProvider
from social_media.reasoner.incremental_reasoner import IncrementalReasoner
from social_media.classifier.classifier_codomain import ClassifierCodomain
from social_media.classifier.mock_classifier import MockClassifier

from social_media.domain.discourse_element import DiscourseElement
from social_media.domain.user import User

from llms_kgs.llms import Gemma3_12B

import datetime as dt

import pprint

Instanciamos a un LLM y construimos al clasificador de elementos de discurso.

In [3]:
gemma = Gemma3_12B(temperature = 0.5, schema = ClassifierCodomain.model_json_schema())

discourse_element_classifier = DiscourseElementClassifier(llm=gemma,
                                                         system_template=DISCOURSE_ELEMENT_CLASSIFIER_SYSTEM_TEMPLATE)


Instanciamos al razonador sobre flujos, el cual tiene como fecha de referencia el día 1 de julio de 2026

In [4]:
reasoner = IncrementalReasoner(classifier = discourse_element_classifier,
                               reference_date = dt.datetime(year=2026,month=7,day=1), win_size=30)

Creamos los elementos de discurso.

In [5]:
element0 = DiscourseElement(
    content = "I dislike everything about myself and I want to harm my arm",
    creation_date = dt.datetime(year=2026, month=7, day=2),
    discourse_id = "0",
    sender = User("0"),
    answers_to = None)

element1 = DiscourseElement(
    content = "I am feeling sad and I do not want to get up to eat",
    creation_date = dt.datetime(year=2026, month=7, day=3),
    discourse_id = "1",
    sender = User("0"),
    answers_to = None)

element2 = DiscourseElement(
    content = "I understand you. My psychologist diagnosed me with depression and I am feeling like a failure",
    creation_date = dt.datetime(year=2026, month=7, day=3),
    discourse_id = "2",
    sender = User("1"),
    answers_to = element1)

element3 = DiscourseElement(
    content = "Today is a happy day and I want to smile to everyone",
    creation_date = dt.datetime(year=2026, month=7, day=4),
    discourse_id = "3",
    sender = User("2"),
    answers_to = None)

Solicitamos al razonador que interprete los elementos de discurso.

In [6]:
reasoner.interpret_discourse_element(element0)
reasoner.interpret_discourse_element(element1)
reasoner.interpret_discourse_element(element2)
reasoner.interpret_discourse_element(element3)
pprint.pprint(reasoner.get_evaluation_function())

{1: ['depressed(0, 0)'], 2: ['depressed(0, 1)', 'depressed(1, 2)']}


Solicitamos al razonador que emita una alarma por los usuarios en riesgo.

In [7]:
alarming_users = reasoner.get_alarming_users(dt.datetime(year=2026,month=7,day=8))

Imprimimos los resultados y las explicaciones para que el experto puede interpretar las alarmas.

In [8]:
for user, diagnostic_list in alarming_users:
    print("In the last month two messages triggered the alarm for the user with ID " + user.get_id())
    for diagnostic_data in diagnostic_list:
        print("<Message>")
        print(diagnostic_data.discourse_element.get_content())
        print("<Analysis>")
        print(diagnostic_data.classifier_result.explanation)
    print("="*80)

In the last month two messages triggered the alarm for the user with ID 0
<Message>
I dislike everything about myself and I want to harm my arm
<Analysis>
The user expresses significant self-dislike and suicidal ideation (desire to harm oneself). Statements like "I dislike everything about myself" indicate deep negative self-perception, low self-esteem, and potential feelings of worthlessness – all common symptoms associated with depression. The explicit mention of wanting to harm oneself is a serious red flag indicating active suicidal thoughts and an immediate risk. This requires urgent attention and intervention.  The intensity and directness of these statements strongly suggest a positive classification for depression.
<Message>
I am feeling sad and I do not want to get up to eat
<Analysis>
The user expresses feelings of sadness and a lack of motivation (not wanting to get up to eat). While experiencing sadness occasionally is normal, the combination of these two factors suggests a

Ahora cargaremos el lote de datos completo y usaremos a un clasificador de juguete para probar el funcionamiento del razonador con un gran volumen de datos.

In [13]:
discourse_elements = DiscourseElementProvider.transform_dataset_to_discourse_element_list('./social-media-streams/mock_dataset')
print(len(discourse_elements))


24439


Imprimimos algunos de los mensajes.

In [14]:
for discourse_element in discourse_elements[0:15]:
    print(discourse_element._creation_date)
    print(discourse_element._content)

2012-12-10 00:00:00
Favourite Che Quote?
I have several. These are my top three:

1. It is better to die standing than to live on your knees.

2. Let me say at the risk of seeming ridiculous that the true revolutionary is guided by great feelings of love.

3.I am not a liberator. Liberators do not exist. The people liberate themselves.

2012-12-11 00:00:00
All fantastic. I think in that first one, he was paraphrasing Zapata, who said something similar, and bringing the quotation back to apply to a new struggle of the same spirit.

**


*If you tremble with indignation at every injustice, then you are a comrade of mine.*

**

*The life of a single human being is worth a million times more than all the property of the richest man on earth.*

**


*I knew that the moment the great governing spirit strikes the blow to divide all humanity into just two opposing factions, I would be on the side of the common people.*
2012-12-11 00:00:00
"Above all, always be capable of feeling deeply any inj

Construimos un clasificador y un razonador de juguete.

In [15]:
mock_classifier = MockClassifier(0.5)

mock_reasoner = IncrementalReasoner(classifier = mock_classifier, reference_date = dt.datetime(year=2010,month=1,day=19), win_size = 365)

Procesamos todos los elementos de discurso.

In [16]:
for discourse_element in discourse_elements:
    mock_reasoner.interpret_discourse_element(discourse_element)

Evaluamos el flujo respecto al 1 de mayo de 2020.

In [17]:
alarming_users = mock_reasoner.get_alarming_users(dt.datetime(year=2020,month=5,day=1))

Imprimimos los resultados.

In [18]:
for user, diagnostic_list in alarming_users[:5]:
    print("In the last 365 days two messages triggered the alarm for the user with ID " + user.get_id())
    for diagnostic_data in diagnostic_list:
        print("<Message>")
        print(diagnostic_data.discourse_element.get_content())
        print("<Analysis>")
        print(diagnostic_data.classifier_result.explanation)
    print("="*80)

In the last 365 days two messages triggered the alarm for the user with ID deleted_user
<Message>
Don't look for a reason, create one
<Analysis>
Obcaecati dolor aliquam incidunt fugit molestiae, maiores sit iusto magni dolorum quia tempore hic?
<Message>
It brings me so much joy to hear this! Hope things keep getting better for you and wishing you the best in life ❤️
<Analysis>
Quibusdam commodi fugit dolore enim in repellat cum libero deserunt, corporis veniam nulla veritatis nihil odit error, ducimus necessitatibus omnis sunt deleniti eligendi repellat, laudantium deleniti obcaecati.
In the last 365 days two messages triggered the alarm for the user with ID deleted_user
<Message>
Thank you for this I really needed to read this. I hope I can make a 180 on my life like you have cause I’ve been feeling absolutely worthless and pointless lately.
<Analysis>
Aut exercitationem odit voluptate unde eum eos sunt, distinctio numquam suscipit, iusto non odio.
<Message>
[removed]
<Analysis>
Culp